In [ ]:
%cd pytorch-openpose

In [ ]:
import cv2
import matplotlib.pyplot as plt
import copy
import numpy as np

import pylab as plt
import time
import os
import argparse
import pandas as pd
import csv

from src import model
from src import util
from src.body import Body
from src.hand import Hand

# Tensorflow libraries
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from keras.models import load_model

# YOLOv4 Setup

In [ ]:
labelsPath = 'coco.names'
LABELS = open(labelsPath).read().strip().split("\n")

In [ ]:
weightsPath = 'yolov4.weights'
configPath = 'yolov4.cfg'

In [ ]:
net = cv2.dnn.readNetFromDarknet(configPath, weightsPath)

# initialize a list of colors to represent each possible class label
np.random.seed(42)
COLORS = np.random.randint(0, 255, size=(len(LABELS), 3),
	dtype="uint8")

In [ ]:
# determine only the *output* layer names that we need from YOLO
ln = net.getLayerNames()
ln = [ln[i - 1] for i in net.getUnconnectedOutLayers()]

# OpenPose Setup

In [ ]:
body_estimation = Body('body_pose_model.pth')
# hand_estimation = Hand('hand_pose_model.pth')

In [ ]:
BODY_PARTS = { "Nose": 0, "Neck": 1, "RShoulder": 2, "RElbow": 3, "RWrist": 4,
               "LShoulder": 5, "LElbow": 6, "LWrist": 7, "RHip": 8, "RKnee": 9,
               "RAnkle": 10, "LHip": 11, "LKnee": 12, "LAnkle": 13, "REye": 14,
               "LEye": 15, "REar": 16, "LEar": 17, "Background": 18 }

POSE_PAIRS = [ ["Neck", "RShoulder"], ["Neck", "LShoulder"], ["RShoulder", "RElbow"],
               ["RElbow", "RWrist"], ["LShoulder", "LElbow"], ["LElbow", "LWrist"],
               ["Neck", "RHip"], ["RHip", "RKnee"], ["RKnee", "RAnkle"], ["Neck", "LHip"],
               ["LHip", "LKnee"], ["LKnee", "LAnkle"], ["Neck", "Nose"], ["Nose", "REye"],
               ["REye", "REar"], ["Nose", "LEye"], ["LEye", "LEar"] ]

threshold=0.5

# Inference

In [ ]:
import math
# draw the body keypoint and libms
def draw_bodypose(canvas, candidate, subset, filter):
    stickwidth = 4
    limbSeq = [[2, 3], [2, 6], [3, 4], [4, 5], [6, 7], [7, 8], [2, 9], [9, 10], \
               [10, 11], [2, 12], [12, 13], [13, 14], [2, 1], [1, 15], [15, 17], \
               [1, 16], [16, 18], [3, 17], [6, 18]]

    colors = [[255, 0, 0], [255, 85, 0], [255, 170, 0], [255, 255, 0], [170, 255, 0], [85, 255, 0], [0, 255, 0], \
              [0, 255, 85], [0, 255, 170], [0, 255, 255], [0, 170, 255], [0, 85, 255], [0, 0, 255], [85, 0, 255], \
              [170, 0, 255], [255, 0, 255], [255, 0, 170], [255, 0, 85]]
    
    points = {}
    
    for i in range(7):
        for n in range(len(subset)):
            index = int(subset[n][i])
            if index == -1:
                continue
            x, y = candidate[index][0:2]
            if i not in filter:
                continue
            cv2.circle(canvas, (int(x), int(y)), 4, colors[i], thickness=-1)
            # cv2.putText(canvas, str(i), (int(x), int(y)), cv2.FONT_HERSHEY_SIMPLEX, 0.3, (0, 0, 0), lineType=cv2.LINE_AA)
            points[i] = (int(x), int(y))
            
    for i in range(7):
        for n in range(len(subset)):
            index = subset[n][np.array(limbSeq[i]) - 1]
            if -1 in index:
                continue
            cur_canvas = canvas.copy()
            Y = candidate[index.astype(int), 0]
            X = candidate[index.astype(int), 1]
            mX = np.mean(X)
            mY = np.mean(Y)
            length = ((X[0] - X[1]) ** 2 + (Y[0] - Y[1]) ** 2) ** 0.5
            angle = math.degrees(math.atan2(X[0] - X[1], Y[0] - Y[1]))
            polygon = cv2.ellipse2Poly((int(mY), int(mX)), (int(length / 2), stickwidth), int(angle), 0, 360, 1)
            cv2.fillConvexPoly(cur_canvas, polygon, colors[i])
            canvas = cv2.addWeighted(canvas, 0.4, cur_canvas, 0.6, 0)
    # plt.imsave("preview.jpg", canvas[:, :, [2, 1, 0]])
    # plt.imshow(canvas[:, :, [2, 1, 0]])
    
    return canvas, points

In [ ]:
# filter = [17, 16, 14, 15, 0, 2 , 1 , 5, 3, 6, 4, 7]
filter = [0, 1, 2, 3, 4, 5, 6, 7]

In [ ]:
# headers = []

# for key in filter:
#     [headers.append(f"x{key}"), headers.append(f"y{key}")]

# pose_header = pd.DataFrame(columns=[headers])
# pose_header

In [ ]:
pose_classifier = tf.keras.models.load_model(r"D:\Projects\Binus\python\PoseDetection\ThesisClean\pytorch-openpose\pose_classifier.h5")

# CSV Creator

# DIFF MODE

In [ ]:
csv_base_filename = "baseline_csv_result"
csv_directory = "csv_result_directory"

In [ ]:
# Create directory if not exist
if not os.path.exists(csv_directory):
    os.makedirs(csv_directory)

In [ ]:
csv_path = r"D:\Projects\Binus\python\PoseDetection\ThesisClean\csv_result_directory\baseline_csv_diff_7.csv"

In [ ]:
header = ["actual_class", "bored", "engaged", "on_the_phone", "raise_hand"]

if not os.path.exists(csv_path):
    with open(csv_path, mode='w', newline='') as f:
        csv_writer = csv.writer(f, delimiter=',', quotechar='"', quoting=csv.QUOTE_MINIMAL)
        csv_writer.writerow(header)
        
else:
    print("CSV exists")

In [ ]:
# Result logger

actual_class = "engaged"

# Multi-Person Estimation

In [ ]:
# Initialize the webcam
cap = cv2.VideoCapture(r"D:\Projects\Binus\python\PoseDetection\ThesisClean\TestVid\4people\4ppl_bored_2_480.mp4")
# cap = cv2.VideoCapture(0)  # 0 is the default camera (you can change it if you have multiple cameras)

while True:
    ret, frame = cap.read()
    if not ret:
        print('Error reading frame from the webcam!')
        break

    (H, W) = frame.shape[:2]

    blob = cv2.dnn.blobFromImage(frame, 1 / 255.0, (416, 416),
        swapRB=True, crop=False)
    net.setInput(blob)
    start = time.time()
    layerOutputs = net.forward(ln)

    boxes = []
    confidences = []
    classIDs = []

    # loop over each of the layer outputs
    for output in layerOutputs:
        # loop over each of the detections
        for detection in output:
            # extract the class ID and confidence (i.e., probability) of
            # the current object detection
            scores = detection[5:]
            classID = np.argmax(scores)
            confidence = scores[classID]

            # filter out weak predictions by ensuring the detected
            # probability is greater than the minimum probability
            if classID == 0:     
                if confidence > 0.5:
                    # scale the bounding box coordinates back relative to the
                    # size of the image, keeping in mind that YOLO actually
                    # returns the center (x, y)-coordinates of the bounding
                    # box followed by the boxes' width and height
                    box = detection[0:4] * np.array([W, H, W, H])
                    (centerX, centerY, width, height) = box.astype("int")

                    # use the center (x, y)-coordinates to derive the top and
                    # and left corner of the bounding box
                    x = int(centerX - (width / 2))
                    y = int(centerY - (height / 2))

                    # update our list of bounding box coordinates, confidences,
                    # and class IDs
                    boxes.append([x, y, int(width), int(height)])
                    confidences.append(float(confidence))
                    classIDs.append(classID)

    idxs = cv2.dnn.NMSBoxes(boxes, confidences, 0.5, 0.3)
    
    # loop over the indexes we are keeping
    if len(idxs) > 0:

        # candidate2, subset2 = body_estimation(frame)
        # canvas2 = copy.deepcopy(frame)
        # pose_inference2 = draw_bodypose(canvas2, candidate2, subset2, filter)

        for i in idxs.flatten():
            # extract the bounding box coordinates
            (x, y) = (boxes[i][0], boxes[i][1])
            (w, h) = (boxes[i][2], boxes[i][3])

            # extract the region of interest (ROI) from the frame based on YOLO bounding box
            roi = frame[y:y+h, x:x+w]

            data = []

            #OpenPose time, pizza time
            candidate, subset = body_estimation(roi)
            canvas = copy.deepcopy(roi)
            pose_inference, points = draw_bodypose(canvas, candidate, subset, filter)
        
            # print("Keypoints:", points)
            # print("Canvas shape:", canvas.shape)

            x_coords = {key: [] for key in filter}
            y_coords = {key: [] for key in filter}

            data.append(points)

            # print(points)
            
            for keypoints in data:
                for key in filter:
                    if key in keypoints:
                        x_coords[key].append(keypoints[key][0])
                        y_coords[key].append(keypoints[key][1])
                    else:
                        x_coords[key].append(-1)
                        y_coords[key].append(-1)
                
                num_keypoints = len(data)
                for i in range(num_keypoints):
                    row = []
                    for key in filter:
                        row.append(x_coords[key][i])
                        row.append(y_coords[key][i])
        
                    # Pose classifier
                    X = pd.DataFrame([row])

                body_language_predict = pose_classifier.predict(X)[0]
                
                predicted_probabilities = body_language_predict
                predicted_class = np.argmax(body_language_predict)

                if predicted_class == 0:
                    predicted_class = 'raise_hand'
                elif predicted_class == 1:
                    predicted_class = 'bored'
                elif predicted_class == 2:
                    predicted_class = 'on_the_phone'
                elif predicted_class == 3:
                    predicted_class = 'raise_hand'
                    
                print(predicted_probabilities)
                print(predicted_class)

                predicted_probabilities[0] = round(predicted_probabilities[0], 4)
                predicted_probabilities[1] = round(predicted_probabilities[1], 4)
                predicted_probabilities[2] = round(predicted_probabilities[2], 4)
                predicted_probabilities[3] = round(predicted_probabilities[3], 4)

                row = [actual_class, predicted_probabilities[0], predicted_probabilities[1],
                       predicted_probabilities[2], predicted_probabilities[3]]    

                print(row)
                
                # Export predictions to CSV
                with open(csv_path, mode='a', newline='') as f:
                    csv_writer = csv.writer(f, delimiter=',', quotechar='"', quoting=csv.QUOTE_MINIMAL)
                    csv_writer.writerow(row)

            # Draw YOLO bounding box on the original frame
            color = [int(c) for c in COLORS[classIDs[i]]]
            cv2.rectangle(frame, (x, y), (x + w, y + h), color, 2)
            text = "{:.4f}".format(confidences[i])
            cv2.putText(frame, text, (x, y - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

            # Display OpenPose inference on the original frame
            frame[y:y + h, x:x + w] = pose_inference

    cv2.imshow("YOLO Object Detection", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
cap.release()
cv2.destroyAllWindows()

In [ ]:
# pose_header

In [ ]:
# row

In [ ]:
# pose_header.loc[len(pose_header)] = row
# pose_header